# Build the dataset: Stats SA QLFS microdata to `qlfs_all.csv`

Run the cells from top to bottom. This notebook:

1. downloads the Quarterly Labour Force Survey (QLFS) microdata zips from Stats SA's ISIbalo portal (or uses zips you saved by hand),
2. turns each quarter into weighted unemployment counts and rates for several breakdowns,
3. combines all quarters into `data/processed/qlfs_all.csv`, the only file Power BI needs.

Raw data is not committed to GitHub (see `.gitignore`). Source: Statistics South Africa, QLFS. Own calculations.

In [ ]:
import re, shutil, zipfile
from pathlib import Path

import pandas as pd
import requests

MICRO = Path("../data/raw/microdata")        # raw Stats SA zips (git-ignored)
OUT = Path("../data/processed/quarters")     # one processed file per quarter
FINAL = Path("../data/processed/qlfs_all.csv")
for p in (MICRO, OUT, FINAL.parent):
    p.mkdir(parents=True, exist_ok=True)

FIRST_YEAR = 2008
LAST = (2026, 2)        # latest published quarter as (year, quarter); change when Stats SA publishes a new one

In [ ]:
# Lookups: codes in the microdata -> labels
PROV = {1: "Western Cape", 2: "Eastern Cape", 3: "Northern Cape", 4: "Free State",
        5: "KwaZulu-Natal", 6: "North West", 7: "Gauteng", 8: "Mpumalanga", 9: "Limpopo"}
GENDER = {1: "Men", 2: "Women"}

# Education_Status (derived by Stats SA) grouped into four groups that are consistent across years
EDU = {1: "Less than matric", 2: "Less than matric", 3: "Less than matric", 4: "Less than matric",
       5: "Matric", 6: "Tertiary", 7: "Other"}

AGE_LABELS = ["15-24", "25-34", "35-44", "45-54", "55-64"]

# Which breakdowns to compute for every quarter
COMBOS = {
    "national": [], "province": ["province"], "age_band": ["age_band"],
    "gender": ["gender"], "education": ["education"],
    "province_age": ["province", "age_band"], "age_education": ["age_band", "education"],
}
DIMS = ["province", "age_band", "gender", "education"]

# Published headline unemployment rates, used as an automatic check on each quarter
KNOWN = {(2019, 1): 27.6, (2019, 4): 29.1, (2020, 1): 30.1, (2025, 1): 32.9,
         (2025, 2): 33.2, (2025, 3): 31.9, (2025, 4): 31.4, (2026, 1): 32.7,
         (2026, 2): 33.6}

In [ ]:
def read_micro(path):
    try:
        return pd.read_csv(path, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, low_memory=False, encoding="latin-1")


def col(df, *names):
    """Find a column by name, ignoring case."""
    lower = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in lower:
            return lower[n.lower()]
    raise KeyError(f"None of {names} found. First columns: {list(df.columns)[:30]}")


def process_quarter(csv_path, year, quarter):
    """One raw QLFS file -> weighted employed/unemployed counts (thousands) and rates by breakdown."""
    df = read_micro(csv_path)
    edu_col = next((c for c in df.columns if c.lower() == "education_status"), None)

    d = pd.DataFrame({
        "age": pd.to_numeric(df[col(df, "Q14AGE")], errors="coerce"),
        "gender": pd.to_numeric(df[col(df, "Q13GENDER")], errors="coerce").map(GENDER),
        "province": pd.to_numeric(df[col(df, "Province")], errors="coerce").map(PROV),
        "status": pd.to_numeric(df[col(df, "Status", "Lfs_Status")], errors="coerce"),
        "education": (pd.to_numeric(df[edu_col], errors="coerce").map(EDU)
                      if edu_col else float("nan")),
        "w": pd.to_numeric(df[col(df, "Weight")], errors="coerce"),
    })

    # Working age (15-64), in the labour force: status 1 = employed, 2 = unemployed
    d = d[d["age"].between(15, 64) & d["status"].isin([1, 2])].copy()
    d["age_band"] = pd.cut(d["age"], [14, 24, 34, 44, 54, 64], labels=AGE_LABELS).astype(str)
    d["emp"] = d["w"].where(d["status"] == 1, 0)
    d["unemp"] = d["w"].where(d["status"] == 2, 0)

    parts = []
    for cut, dims in COMBOS.items():
        if dims:
            agg = d.groupby(dims).agg(emp=("emp", "sum"), unemp=("unemp", "sum"),
                                      sample_n=("w", "size")).reset_index()
        else:
            agg = pd.DataFrame([{"emp": d["emp"].sum(), "unemp": d["unemp"].sum(),
                                 "sample_n": len(d)}])
        agg["cut"] = cut
        parts.append(agg)

    res = pd.concat(parts, ignore_index=True)
    for c in DIMS:
        if c not in res.columns:
            res[c] = "All"
        res[c] = res[c].fillna("All")

    res["unemployment_rate"] = (100 * res["unemp"] / (res["emp"] + res["unemp"])).round(2)
    res["employed_k"] = (res["emp"] / 1000).round(3)
    res["unemployed_k"] = (res["unemp"] / 1000).round(3)
    res.insert(0, "quarter", quarter)
    res.insert(0, "year", year)

    nat = res.loc[res["cut"] == "national", "unemployment_rate"].iloc[0]
    cols = ["year", "quarter", "cut"] + DIMS + ["employed_k", "unemployed_k",
                                                "unemployment_rate", "sample_n"]
    return res[cols], nat

In [ ]:
BASE = "https://isibaloweb.statssa.gov.za/data/PSS/HSS/QLFS/{y}/QLFS{y}{q:02d}.zip"

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
})


def fetch(y, q):
    page = f"https://isibaloweb.statssa.gov.za/pages/surveys/pss/qlfs/{y}/qlfs{y}.php"
    SESSION.get(page, timeout=60)          # visit the page first, like a browser would
    return SESSION.get(BASE.format(y=y, q=q), headers={"Referer": page}, timeout=300)


def get_quarter(y, q, keep_raw=True):
    out_file = OUT / f"qlfs_{y}Q{q}.csv"
    if out_file.exists():                  # already processed, skip
        return pd.read_csv(out_file), "cached"

    zpath = MICRO / f"QLFS{y}{q:02d}.zip"
    if not zpath.exists():                 # only download if the zip is not already on disk
        try:
            r = fetch(y, q)
        except requests.RequestException as e:
            return None, f"download error: {e}"
        if r.status_code != 200 or r.content[:2] != b"PK":
            return None, f"download failed (status {r.status_code}); save {zpath.name} by hand"
        zpath.write_bytes(r.content)

    folder = MICRO / f"QLFS{y}{q:02d}_x"
    try:
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(folder)
        csvs = list(folder.rglob("*.csv"))
        if not csvs:
            return None, "no csv inside zip"
        res, nat = process_quarter(csvs[0], y, q)
    except Exception as e:
        return None, f"processing error: {e}"
    finally:
        shutil.rmtree(folder, ignore_errors=True)
        if not keep_raw:
            zpath.unlink(missing_ok=True)

    res.to_csv(out_file, index=False)
    return res, "ok"


def run(years, last=LAST):
    for y in years:
        for q in (1, 2, 3, 4):
            if (y, q) > last:
                continue
            res, status = get_quarter(y, q)
            if res is None:
                print(f"{y} Q{q}: {status}")
                continue
            nat = res.loc[res.cut == "national", "unemployment_rate"].iloc[0]
            pub = KNOWN.get((y, q))
            flag = "" if pub is None else ("  CHECK OK" if abs(nat - pub) <= 0.1 else f"  MISMATCH (published {pub})")
            print(f"{y} Q{q}: national {nat}{flag}  [{status}]")

## Download and process every quarter

Quarters already processed are skipped, so it is safe to run this cell again. If a download fails with a 403,
save that quarter's zip by hand from the ISIbalo year page into `data/raw/microdata/` (keep the file name, e.g. `QLFS202601.zip`) and run the cell again.

In [ ]:
run(range(FIRST_YEAR, LAST[0] + 1), last=LAST)

## Combine into the file Power BI loads

In [ ]:
files = sorted(OUT.glob("qlfs_*Q*.csv"))
all_q = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

all_q["quarter_end"] = (pd.to_datetime(all_q["year"].astype(str) + "-" +
                        (all_q["quarter"] * 3).astype(str) + "-01") + pd.offsets.MonthEnd(0))
all_q["period"] = all_q["year"].astype(str) + " Q" + all_q["quarter"].astype(str)
all_q["low_sample"] = all_q["sample_n"] < 150    # rule of thumb for noisy cells, not a Stats SA standard

# Checks
key = ["year", "quarter", "cut", "province", "age_band", "gender", "education"]
assert not all_q.duplicated(key).any(), "duplicate rows found"

have = set(zip(all_q["year"], all_q["quarter"]))
expected = {(y, q) for y in range(FIRST_YEAR, LAST[0] + 1) for q in (1, 2, 3, 4) if (y, q) <= LAST}
print("Quarters:", len(have), "| expected:", len(expected), "| missing:", sorted(expected - have))

recomputed = 100 * all_q["unemployed_k"] / (all_q["employed_k"] + all_q["unemployed_k"])
print("Largest gap between stored and recomputed rate:", round((recomputed - all_q["unemployment_rate"]).abs().max(), 3))

all_q.to_csv(FINAL, index=False)
print("Saved", FINAL, "|", all_q.shape)